# 07 — ฟังก์ชัน และ Module

**คอร์ส Python for Geography** · ภาควิชาภูมิศาสตร์ ชั้นปีที่ 2
**วันที่ 4 · ช่วงที่ 2 · เวลาโดยประมาณ 60 นาที**

---

### สิ่งที่จะได้จากบทนี้
- เขียนฟังก์ชันด้วย `def` มี **parameter**, **return**, **default value**
- เขียน **docstring** อธิบายฟังก์ชันให้คนอื่น (และตัวเองในอีก 3 เดือน) อ่านรู้เรื่อง
- เข้าใจ **scope** ของตัวแปร
- `import` module มาตรฐาน และ **สร้าง module ของตัวเอง** เก็บไว้ใช้ซ้ำ

> **วิธีเรียน:** โค้ดในเซลล์สีเทาคือโค้ดที่อาจารย์จะรันให้ดู
> ใต้แต่ละหัวข้อจะมีเซลล์ ✍️ ว่างไว้ — **ให้พิมพ์ตามด้วยตัวเอง ห้าม copy–paste**
> การพิมพ์เองคือวิธีเดียวที่จะจำ syntax ได้

## 1. ทำไมต้องมีฟังก์ชัน

ถ้าต้องจำแนกระดับฝนใน 5 ที่ของโปรแกรม แล้วเกณฑ์เปลี่ยน เราต้องไล่แก้ 5 ที่
ถ้าเขียนเป็นฟังก์ชัน แก้ที่เดียวจบ — นี่คือหลักการ **DRY (Don't Repeat Yourself)**

```python
def ชื่อฟังก์ชัน(parameter):
    """docstring อธิบายว่าทำอะไร"""
    คำสั่ง
    return ค่าที่ส่งกลับ
```

In [ ]:
def mm_to_inch(mm):
    """แปลงปริมาณฝนจากมิลลิเมตรเป็นนิ้ว"""
    return mm / 25.4

print(mm_to_inch(45.5))
print(f"{mm_to_inch(118.6):.2f} inch")
print(mm_to_inch.__doc__)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

### `return` กับ `print` ไม่เหมือนกัน

- `print` = แสดงให้คนดู ส่งค่ากลับไม่ได้
- `return` = ส่งค่ากลับไปให้โปรแกรมใช้ต่อ

ฟังก์ชันที่ไม่มี `return` จะคืนค่า `None` — เป็นสาเหตุของบั๊ก "ทำไมได้ None" ที่เจอบ่อย

In [ ]:
def bad_convert(mm):
    print(mm / 25.4)      # แค่แสดงผล

def good_convert(mm):
    return mm / 25.4

a = bad_convert(45.5)
b = good_convert(45.5)
print("a =", a)
print("b =", b)
print("คำนวณต่อได้:", b * 2)

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 2. หลาย parameter, default value, keyword argument

In [ ]:
def rain_volume(area_km2, rain_mm, runoff_ratio=1.0):
    """
    คำนวณปริมาตรน้ำฝนที่ตกลงในพื้นที่รับน้ำ

    area_km2      : พื้นที่รับน้ำ (ตารางกิโลเมตร)
    rain_mm       : ปริมาณฝน (มิลลิเมตร)
    runoff_ratio  : สัดส่วนน้ำท่า 0-1 (ค่าเริ่มต้น 1.0 = ไม่คิดการซึม)

    คืนค่า: ปริมาตร (ลูกบาศก์เมตร)
    """
    return area_km2 * 1_000_000 * (rain_mm / 1000) * runoff_ratio

print(rain_volume(12.5, 45))                       # ใช้ค่า default
print(rain_volume(12.5, 45, 0.6))                  # ส่งตามตำแหน่ง
print(rain_volume(area_km2=12.5, rain_mm=45, runoff_ratio=0.6))  # keyword ชัดกว่า
print(f"{rain_volume(12.5, 118.6, 0.6):,.0f} m3")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 3. Scope — ตัวแปรในฟังก์ชันอยู่แค่ในฟังก์ชัน

In [ ]:
threshold = 35.0          # global

def is_heavy(mm):
    local_note = "ตรวจสอบแล้ว"    # local อยู่แค่ในฟังก์ชัน
    return mm > threshold          # อ่าน global ได้

print(is_heavy(45.5))
print(threshold)
print(local_note)     # <- error: ตัวแปร local ไม่มีอยู่ข้างนอก

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

> **คำแนะนำ:** อย่าให้ฟังก์ชันพึ่ง global มากเกินไป ส่งค่าที่ต้องใช้เข้าไปเป็น parameter
> จะทดสอบง่ายกว่าและย้ายไปใช้ที่อื่นได้ทันที

## 4. ฟังก์ชันที่ใช้จริงในงานภูมิศาสตร์: `haversine`

ระยะทางระหว่างสองพิกัดบนผิวโลก (great-circle distance) — สูตรนี้ถูกต้องกว่าการใช้ Pythagoras
เพราะคิดความโค้งของโลก

In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    """ระยะทางวงกลมใหญ่ระหว่างสองพิกัด (กิโลเมตร)"""
    R = 6371.0                      # รัศมีโลกเฉลี่ย (km)
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1) * math.cos(p2) * math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

d = haversine(13.367, 100.983, 13.150, 101.350)
print(f"ชลบุรี -> หนองใหญ่ = {d:.2f} km")

stations = {
    "CHB01": (13.367, 100.983),
    "SRC01": (13.163, 100.803),
    "NYI01": (13.150, 101.350),
    "STH01": (12.680, 100.983),
}
origin = stations["CHB01"]
for sid, (lat, lon) in stations.items():
    print(f"CHB01 -> {sid}: {haversine(origin[0], origin[1], lat, lon):7.2f} km")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 5. `import` module มาตรฐาน

In [ ]:
import math
import random
import datetime as dt

print(math.pi, math.sqrt(2), math.radians(180))

random.seed(2025)                     # ตั้ง seed ให้ผลลัพธ์ซ้ำได้
print([round(random.uniform(0, 50), 1) for _ in range(5)])

today = dt.date(2025, 9, 14)
print(today, today.month, today.strftime("%d/%m/%Y"))
print("ห่างจากต้นปี", (today - dt.date(2025, 1, 1)).days, "วัน")

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 6. สร้าง module ของตัวเอง

`%%writefile` คือคำสั่งพิเศษของ Colab ที่เขียนเนื้อหาในเซลล์ลงไฟล์
เราใช้มันสร้างไฟล์ `.py` ของตัวเอง แล้ว `import` เหมือน module ทั่วไป

**นี่คือวิธีที่ทำให้โค้ดถูกนำไปใช้ซ้ำได้จริง** — notebook ใช้เล่าเรื่อง ส่วน `.py` เก็บ "กระบวนการ"

In [ ]:
%%writefile my_geo.py
"""ฟังก์ชันช่วยงานภูมิศาสตร์ของฉัน"""
import math


def mm_to_inch(mm):
    """แปลง mm เป็นนิ้ว"""
    return mm / 25.4


def classify_rain(mm):
    """จำแนกระดับฝนรายวันตามเกณฑ์ 24 ชั่วโมง"""
    if mm < 0.1:
        return "ไม่มีฝน"
    if mm <= 10.0:
        return "ฝนเล็กน้อย"
    if mm <= 35.0:
        return "ฝนปานกลาง"
    if mm <= 90.0:
        return "ฝนหนัก"
    return "ฝนหนักมาก"


def haversine(lat1, lon1, lat2, lon2):
    """ระยะทางวงกลมใหญ่ (km)"""
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

In [ ]:
import my_geo

print(my_geo.classify_rain(118.6))
print(my_geo.mm_to_inch(45.5))
print(round(my_geo.haversine(13.367, 100.983, 13.150, 101.350), 2))

# import เฉพาะบางฟังก์ชัน หรือใช้ชื่อย่อ
from my_geo import classify_rain
import my_geo as gu

print(classify_rain(5.0), gu.classify_rain(50.0))

In [ ]:
# ✍️ พิมพ์โค้ดด้านบนด้วยตัวเองที่นี่

## 7. แบบฝึกหัดท้ายบท

1. เขียนฟังก์ชัน `c_to_f(c)` และ `f_to_c(f)` พร้อม docstring แล้วทดสอบไป–กลับว่าได้ค่าเดิม
2. เขียนฟังก์ชัน `rain_summary(values)` ที่รับ list ของปริมาณฝน แล้ว **return dict**
   ที่มี key: `total`, `mean`, `max`, `rain_days`
3. เขียนฟังก์ชัน `nearest_station(lat, lon, stations)` คืน `(station_id, ระยะทาง_km)` ของสถานีที่ใกล้ที่สุด
4. เพิ่มฟังก์ชันทั้งหมดข้างบนลงไฟล์ `my_geo.py` (เขียนทับด้วย `%%writefile`) แล้ว `import` มาใช้ใหม่

In [ ]:
# ข้อ 1

In [ ]:
# ข้อ 2

In [ ]:
# ข้อ 3
stations = {
    "CHB01": (13.367, 100.983),
    "SRC01": (13.163, 100.803),
    "NYI01": (13.150, 101.350),
    "STH01": (12.680, 100.983),
}

### เฉลย
```python
# ข้อ 1
def c_to_f(c):
    """แปลงองศาเซลเซียสเป็นฟาเรนไฮต์"""
    return c * 9 / 5 + 32

def f_to_c(f):
    """แปลงองศาฟาเรนไฮต์เป็นเซลเซียส"""
    return (f - 32) * 5 / 9

print(c_to_f(35), f_to_c(c_to_f(35)))

# ข้อ 2
def rain_summary(values):
    """สรุปสถิติฝนจาก list ของค่ารายวัน"""
    return {
        "total": round(sum(values), 1),
        "mean": round(sum(values) / len(values), 2),
        "max": max(values),
        "rain_days": len([v for v in values if v >= 0.1]),
    }

print(rain_summary([0.0, 12.4, 45.5, 118.6, 3.2]))

# ข้อ 3
def nearest_station(lat, lon, stations):
    """หาสถานีที่ใกล้พิกัดที่กำหนดที่สุด"""
    best_id, best_d = None, float("inf")
    for sid, (slat, slon) in stations.items():
        d = haversine(lat, lon, slat, slon)
        if d < best_d:
            best_id, best_d = sid, d
    return best_id, round(best_d, 2)

print(nearest_station(13.20, 101.10, stations))
```